# Imports and hyperparms
Essential imports and hyper params for the code

In [1]:
##########################################################
# Imports
##########################################################
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import torch_directml

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18 

import copy

##########################################################
# Hyper Parameters
##########################################################

# Dataset transformations/augmentations
ROTATION_DEGREES = 10
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
WIDTH = 224
HEIGHT = 224
SIZE = (WIDTH,HEIGHT)
DEVICE = torch_directml.device()                                                        # AMD GPU
# DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')                 # Nvidia GPU 

# TRAINING
NUM_OF_CLASSES = 10
BATCH_SIZE = 64
EPOCHS = 20
EPOCHS_BEFORE_UNFREEZE = 3
LEARNING_RATE = 0.001
LR_DECAY_EPOCS = 7
LR_DECAY_GAMMA = 0.1

# 1. Setup and Data Preperation

In [2]:
# List of classes in the Fashion-MNIST dataset
classes = ('T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle Boot')

# training transoformations
# we augment the data with a horizontal flip, slight rotation to add variation
# and we normalize using mean and standard deviation
transform_train = transforms.Compose([
    transforms.Resize(SIZE),                                # match to size 224x224
    transforms.Grayscale(num_output_channels=3),            # mimic RGB
    transforms.RandomHorizontalFlip(),                      # Randomly flip
    transforms.RandomRotation(ROTATION_DEGREES),            # Randomly rotate
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),       # color jitter
    transforms.ToTensor(),                                  # Convert to PyTorch Tensor
    transforms.Normalize(MEAN, STD)                         # Normalize
])

# Test transformations, no augmentations since we want clean test data
transform_test = transforms.Compose([
    transforms.Resize(SIZE),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])


# download the dataset
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform_train
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform_test
)


# Dataloarders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 2. Model Building 

In [3]:
print(f"Using device: {DEVICE}")

model = resnet18(pretrained=True)           # load pre trained model

for param in model.parameters():            # freeze the weights
    param.requires_grad = False


finalLayerSize = model.fc.in_features                   # number of inputs for final layer   
model.fc = nn.Linear(finalLayerSize, NUM_OF_CLASSES)    # new output layer to fit our number of classes
model = model.to(DEVICE)

# Helper function to later unfreeze weights during training
def unfreeze_model(model):
    for param in model.parameters():
        param.requires_grad = True
    print("Model unfrozen")

Using device: privateuseone:0


c:\Users\YuriM\mamba_envs\deep_learning\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\YuriM\mamba_envs\deep_learning\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


# 3. Model Training

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=LR_DECAY_EPOCS, gamma=LR_DECAY_GAMMA)

best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0

# used later for plots
train_losses = []
val_losses = []
train_accs = []
val_accs = []

print(f"Training mode...")

for epoch in range(EPOCHS):
    print(f'Epoch {epoch+1}/{EPOCHS}')
    print('-' * 10)

    # If reached the threshold and it's time to unfreeze
    if epoch == EPOCHS_BEFORE_UNFREEZE:
        print("Unfreezing all layers for fine-tuning")
        unfreeze_model(model)
        # update the optimizer to include ALL model parameters now
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE * 0.1)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()  # Set model to training mode
            dataloader = train_loader
        else:
            model.eval()   # Set model to evaluate mode
            dataloader = test_loader

        running_loss = 0.0
        all_preds = []
        all_labels = []

        # Iterate over data
        for inputs, labels in dataloader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward
            # Track history only if in train
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                # Backward + optimize only if in training phase
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            # Statistics
            running_loss += loss.item() * inputs.size(0)
            
            # Collect predictions for metrics
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        if phase == 'train':
            scheduler.step()

        # Calculate Metrics
        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_acc = accuracy_score(all_labels, all_preds)
        
        # save for later
        if phase == 'train':
            train_losses.append(epoch_loss)
            train_accs.append(epoch_acc)
        else:
            val_losses.append(epoch_loss)
            val_accs.append(epoch_acc)

        # Calculate Precision, Recall, F1
        epoch_precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
        epoch_recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
        epoch_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} '
              f'Prec: {epoch_precision:.4f} Rec: {epoch_recall:.4f} F1: {epoch_f1:.4f}')

        # Save Best Model
        if phase == 'val' and epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), 'best_resnet_fashionmnist.pth')
            print(f"  -> New best model saved! (Acc: {best_acc:.4f})")


Training mode...
Epoch 1/20
----------


c:\Users\YuriM\mamba_envs\deep_learning\Lib\site-packages\torch\optim\adam.py:534: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  torch._foreach_lerp_(device_exp_avgs, device_grads, 1 - beta1)


Train Loss: 0.7120 Acc: 0.7636 Prec: 0.7615 Rec: 0.7636 F1: 0.7618
Val Loss: 0.5930 Acc: 0.7928 Prec: 0.8035 Rec: 0.7928 F1: 0.7839
  -> New best model saved! (Acc: 0.7928)
Epoch 2/20
----------
Train Loss: 0.5232 Acc: 0.8135 Prec: 0.8121 Rec: 0.8135 F1: 0.8124
Val Loss: 0.5386 Acc: 0.8104 Prec: 0.8167 Rec: 0.8104 F1: 0.8011
  -> New best model saved! (Acc: 0.8104)
Epoch 3/20
----------
Train Loss: 0.4986 Acc: 0.8216 Prec: 0.8200 Rec: 0.8216 F1: 0.8205
Val Loss: 0.5195 Acc: 0.8160 Prec: 0.8222 Rec: 0.8160 F1: 0.8120
  -> New best model saved! (Acc: 0.8160)
Epoch 4/20
----------
Unfreezing all layers for fine-tuning
Model unfrozen
Train Loss: 0.2846 Acc: 0.8973 Prec: 0.8970 Rec: 0.8973 F1: 0.8971
Val Loss: 0.2140 Acc: 0.9227 Prec: 0.9240 Rec: 0.9227 F1: 0.9226
  -> New best model saved! (Acc: 0.9227)
Epoch 5/20
----------
Train Loss: 0.1905 Acc: 0.9301 Prec: 0.9299 Rec: 0.9301 F1: 0.9300
Val Loss: 0.1948 Acc: 0.9306 Prec: 0.9326 Rec: 0.9306 F1: 0.9311
  -> New best model saved! (Acc: 0.

KeyboardInterrupt: 

# 4. Evaluation

In [ ]:
checkpoint_path = 'best_resnet_fashionmnist.pth'
model.load_state_dict(torch.load(checkpoint_path))
model = model.to(DEVICE)
model.eval() # Set to evaluation mode

print(f"Loaded best model weights from {checkpoint_path}")

# Run Inference on Test Set
all_preds = []
all_labels = []

print("Running evaluation on Test Set...")
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute Metrics
test_acc = accuracy_score(all_labels, all_preds)
test_precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
test_recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
test_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print("-" * 30)
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print(f"Test F1 Score:  {test_f1:.4f}")
print("-" * 30)
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=classes))

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - ResNet18 on FashionMNIST')
plt.show()

def plot_training_curves(train_losses, val_losses, train_accs, val_accs):
    epochs = range(1, len(train_losses) + 1)

    plt.figure(figsize=(14, 5))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label='Train Loss', marker='o')
    plt.plot(epochs, val_losses, label='Validation Loss', marker='o')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_accs, label='Train Accuracy', marker='o')
    plt.plot(epochs, val_accs, label='Validation Accuracy', marker='o')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_training_curves(train_losses, val_losses, train_accs, val_accs)